In [1]:
%%time
#this loads the dataset, and verifies its length
lines = open("../data/itf_slice100k.txt").read().splitlines() #this line simply sets my simplified dataset equal to "lines:
print(len(lines)) #prints how many lines, should be 100,000 for itf)slice100k.txt
#after first test, printed the correct number of lines

100000
CPU times: user 10.4 ms, sys: 17.4 ms, total: 27.8 ms
Wall time: 29.5 ms


In [2]:
%%time
#this cell will allow me to approx eyeball the raw structure, as it's a new file
for line in lines[:5]: #initiates a for loop, that will run for the first five lines
    print(repr(line)) #simply prints out everying on the line

'     /7239   4C2015 05 23.30928 11 55 25.17 -01 46 36.9          23.7 z1     T09'
'     /7239   4C2015 05 23.32686 11 55 25.34 -01 46 36.1          23.7 z1     T09'
'     /7239   4C2015 05 23.34140 11 55 25.49 -01 46 35.4          23.5 z1     T09'
'     /2050   4C2016 07 12.54696 23 31 53.92 -00 17 11.8          23.1 z1     T09'
'     /2050   4C2016 07 12.56833 23 31 54.24 -00 17 08.9          22.0 z1     T09'
CPU times: user 814 μs, sys: 0 ns, total: 814 μs
Wall time: 1.23 ms


In [4]:
%%time
#Cell copied over from the parsing of the 3I astrometry. Slightly modified.
import pandas as pd

rows = []
skipped_short = 0 #this creates a counter for lines we couldn't parse
skipped_pair = 0
for line in lines:
    if len(line) < 80: #if the line is shorter than 80
        skipped_short += 1
        continue
    if line[14].islower():
        skipped_pair += 1
        continue
    #the below is a fix to make the parser detect if there is a shift from an S at the start of the date, from if the observer was a space telescope. This should fix that error.
    off = 0 if line[15].isdigit() else 1
    rows.append({ #builds a list for the current line of all of its important info (normalizes data)
        "desig":    line[:12].strip(),
        "date_str": line[15+off:32+off].strip(),
        "ra_str":   line[32+off:44+off].strip(),
        "dec_str":  line[44+off:56+off].strip(),
        "mag_str":  line[65+off:70+off].strip(),
        "obscode":  line[77+off:80+off],
    })

df = pd.DataFrame(rows) #turns the list into a table
print(len(df), "parsed |", skipped_short, "short |", skipped_pair, "continuation") #prints how many rows were skipped, and how many made it to the table
df.head() #shows the first five rows of now parsed data

97746 parsed | 0 short | 2254 continuation
CPU times: user 213 ms, sys: 25.3 ms, total: 238 ms
Wall time: 242 ms


,desig,date_str,ra_str,dec_str,mag_str,obscode
0,/7239,2015 05 23.30928,11 55 25.17,-01 46 36.9,23.7,T09
1,/7239,2015 05 23.32686,11 55 25.34,-01 46 36.1,23.7,T09
2,/7239,2015 05 23.34140,11 55 25.49,-01 46 35.4,23.5,T09
3,/2050,2016 07 12.54696,23 31 53.92,-00 17 11.8,23.1,T09
4,/2050,2016 07 12.56833,23 31 54.24,-00 17 08.9,22.0,T09


In [5]:
%%time
#This cell will measure the memory that all lines take up

gb = df.memory_usage(deep=True).sum() / 1e9 #counts the true cost of everything, includeing strings, it then totals all columns, then converts bytes to GB
print(round(gb, 3), "GB for 100k lines")

0.034 GB for 100k lines
CPU times: user 124 ms, sys: 2.14 ms, total: 126 ms
Wall time: 280 ms


In [7]:
%%time
#this cell will extrapolate based off all the numbers/data i've collected so far, to calculate how long it will take, and how much space it will take up

TOTAL = 9354883 #the total amount of lines
t_slice = 0.544 #the amount of time it took to parse 100k lines
gb_slice = 0.034 #the amount of data used for the 100k lines

scale = TOTAL / 100000 #this determines what we need to scale our current data and time values by
print("projected parse time:", round(t_slice * scale / 60, 1), "minutes") #calculates how long its expected to parse by multipling our current value by the scale factor, and then converting it to minutes, as its currently in milliseconds
print("projected dataframe:", round(gb_slice * scale, 1), "GB") #does the same as above, but for gigabytes 

projected parse time: 0.8 minutes
projected dataframe: 3.2 GB
CPU times: user 276 μs, sys: 0 ns, total: 276 μs
Wall time: 259 μs


In [9]:
%%time
#"flag" census

from collections import Counter #imports python's counter
flags = Counter(line[14] for line in lines if len(line) >= 80) #this parses through all 100000 lines, and adds the charachter at index 14 to the flag variable, whenever the length is above or equal to 80 length
print(flags.most_common(10)) #prints out the 10 most common flags


[('C', 95470), ('S', 2254), ('s', 2254), ('B', 16), (' ', 6)]
CPU times: user 13.3 ms, sys: 813 μs, total: 14.1 ms
Wall time: 15 ms


In [10]:
%%time
#This is a validator cell, to ensure that every weird line gets caught before going to the next few cells
looks_ok = df["ra_str"].str.match(r"^\d{2} \d{2} \d{2}") #this line tests every entry in the ra_str column against a pattern, only passing lines that pass the pattern, testing each invididual row. The pattern itself checks to ensure that starting at the very beginning of the string, there are exactly 3 pairs of two digits seperated by spaces.
print((~looks_ok).sum(), "suspicious rows") #the ~ means not, and flips every true to false and visa versa, so looks ok then marks the rows that failed the test, so we know how many failed without creating a new variable. .sum simply counts how many of these failures there are, and the print tells me the quantity
df[~looks_ok].head() #This line prints out the first five rows that didn't pass the test.

0 suspicious rows
CPU times: user 59.2 ms, sys: 1.47 ms, total: 60.6 ms
Wall time: 147 ms


,desig,date_str,ra_str,dec_str,mag_str,obscode


In [ ]:
%%time
#this cell will preform structural checks
print(df["desig"].value_counts().head(10)) #Will print the "biggest" tracklets in the file
print(df["obscode"].value_counts().head(10)) #Which observatories are in the head of the file
print(df["desig"].nunique(), "distinct designations in file slice") #prints out the number of different objects in this file

desig
0001198    32
0003065    30
0001935    28
0003154    26
0000781    23
0001501    23
06AQUV     23
0001508    22
0000533    21
0023651    21
Name: count, dtype: int64
obscode
705    81145
699    10931
C51     2254
Z18      846
568      568
Z32      468
608      183
J75      156
W92      101
I41       73
Name: count, dtype: int64
45623 distinct designations in file
CPU times: user 57.2 ms, sys: 1.15 ms, total: 58.3 ms
Wall time: 98.5 ms
